« model_20 — TOPLAMA EĞİTİMİ (train_math) · COLAB KOŞULARI »

**Yalnız cok** (kullanici, 27 Eylul: "matematik verisi sadece çok olsun"; dur'un sonuçları yalnız kayıtta, `OLCULENLER_math.md` §1).

**Veri:** model_15'in toplama kümeleri (üretim kodu `deneme2/model_15/veri_dur.py`, `veri_cok.py`; dizi ve ölçüt `deneme2/model_19/data_mat_19.py`; hepsi `data_math.py` / `exam_math.py`'ye KOPYA). **dur**: 2 terim, a, b ∈ 0..500 (251.000 çift), eğitim 126.776 / tutulan 124.224. **cok**: dur'un çiftleri aynen + 251.000 üçlü, eğitim 252.365 / tutulan 249.635 (üçlü 125.589 / 125.411). Dizi `<eos> 4 7 2 + 1 8 2 = 6 5 4 <eos>`, sözlük 13 (0-9 + = `<eos>`), dolgu `<eos>` (maskede yok). Veri koddan üretilir (saniyeler) ve iz model_15'in kaydıyla karşılaştırılır: dur `eb4c73ab05178e18`, cok `8d0f89938c67e0e7`. Yedek: `MyDrive/model_20/train_math/veri_<ad>.pt` (`SOURCE = 'drive'`).
**Ölçüt** model_15 / model_19 ile aynı (`exam_math.ask`): `<eos>`'a kadar serbest açgözlü üretim; accuracy = cevap birebir VE doğru yerde durdu; length_ok; first_digit. Her 5.000 adımda sabit 20.000'lik örnek (tohum 12345, model_15'in günlüğü gibi); sonda BÜTÜN küme, terim ve hane kırılımı (model_15 `kayit.txt` gibi). **Sağlık:** eğitim örneğinde accuracy ≥ 0,99; geçen en küçük boyut seçilir.
**Kayıtlı tabanlar** (Drive `model_15/DUR/kayit.txt`, `COK/kayit.txt`; eski mimari 4.529 parametre, lr 0,002, wd 0,01, yığın 25.000 sonraki-token örneği): dur tutulan 0,7770 (uzunluk 0,9916; 8.000 adım); cok tutulan 0,4403 — 2 terim 0,5121, 3 terim 0,3693 (16.000 adım). model_19 (329.480 parametre, batch 4.096, 20.000 adım): cok tutulan 0,9906 (`belge/OLCULENLER.md` §1l). Model ve tarif farklı: fark atfedilemez.
**Model:** Model X (`SETTING = 'shared'`), D 64 ve 96, units 4·D, 2 tur; `'transformer_novalue'` kıyas transformer'ı (MODEL_KW'de `layers`). Batch 256, 100 epok (dur 49.500 adım, cok 98.500), lr 0,01 cosine + clipping (Model X tarifi).
Karar kuralı ve sonuçlar: `deneme2/model_20/train_math/OLCULENLER_math.md`.

| hücre | iş |
|---|---|
| 0 HAZIRLIK | Drive, kod (GitHub'dan taze), veri koddan (iz karşılaştırılır) |
| 1a / 1b | dur, Model X D 64 / D 96 (öneri: önce bunlar) |
| 1c / 1d | cok, Model X D 64 / D 96 |
| 2 NABIZ | durum ve son satırlar |
| 3 SONUÇ | sağlık, terim/hane kırılımı, tanı soruları, yanlışlar (kural 12) |
| X / Y / Z | durdur · Drive'daki kayıt · GPU durumu |

In [ ]:
# 0 HAZIRLIK  |  CPU  |  tekrar: GUVENLI
# Cekirdek dustuyse ONCE bu hucre.  Veri koddan (saniyeler); iz model_15'in kaydiyla karsilastirilir, tutmazsa DURUR.
import os, sys, subprocess

# Canli kosu varken moduller yeniden yuklenirse kosu listesi sifirlanir.
if 'colab_math' in sys.modules:
    _live = [n for n, r in sys.modules['colab_math'].RUNS.items() if r['thread'].is_alive()]
    assert not _live, 'CEKIRDEK CANLI, kosu suruyor: %s -- HAZIRLIK gerekmiyor, 2 NABIZ' % _live

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/model_20'
DATA_DIR = ROOT + '/train_math'                   # YEDEK: model_15'in veri_dur.pt / veri_cok.pt kopyasi (SOURCE = 'drive')
os.makedirs(ROOT, exist_ok=True)

# Kod her seferinde TAZE cekilir -- Colab'da elle duzenleme birikmesin.
REPO = '/content/sekerai'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '-q', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/sekerahmet/sekerai.git', REPO], check=True)
SRC = REPO + '/deneme2/model_20'                  # Model X, kiyas transformer'i ve genel egitim
MATH = SRC + '/train_math'                        # toplama egitimi: veri, olcut, calistirici
CODE = subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
print('kod   ' + CODE)
assert os.path.exists(MATH + '/colab_math.py'), 'depoda train_math/colab_math YOK -- yerelde git push gerekli'
for _p in (SRC, MATH):
    if _p not in sys.path:
        sys.path.insert(0, _p)
for _m in ('data_math', 'model_20', 'model_20_transformer', 'train_20', 'exam_math', 'colab_math'):
    sys.modules.pop(_m, None)          # taze kod gercekten yuklensin

import torch
import data_math as DM
import train_20 as TR
import exam_math as EM
import colab_math as C

SOURCE = 'code'                  # 'code': model_15'in kodundan uretilir; 'drive': DATA_DIR'deki model_15 dosyasi (ayni iz)
DATASETS = ('cok',)              # YALNIZ cok (kullanici, 27 Eylul: "matematik verisi sadece çok olsun"); dur yalniz eski kayit
DATA = {}
for _name in DATASETS:
    DATA[_name] = DM.build(_name, SOURCE, DATA_DIR + '/veri_%s.pt' % _name)
    DM.audit(DATA[_name])
    print(DM.report(DATA[_name]))

In [ ]:
# 1a KOSU -- Model X, cok, D 64  |  GPU  |  tekrar: RESUME=False iken ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ; RESUME=True iken son checkpoint'ten SURDURUR
DATA_NAME, D = 'cok', 64
assert DATA_NAME == 'cok', "matematik yalniz cok (kullanici, 27 Eylul: 'matematik verisi sadece çok olsun')"
SETTING = 'shared'          # 'transformer_novalue': kiyas transformer'i (o zaman MODEL_KW'de turns yerine layers=2)
SEED = 0
NAME = 'modelx_math_%s_d%d_s%d' % (DATA_NAME, D, SEED)     # ONERI -- ad kullanici onayiyla (kural 11)
MODEL_KW = dict(d=D, units=4 * D, turns=2)
BATCH_SIZE = 256            # kullanici karari
EPOCHS = 100                # kullanici karari
STEPS = EPOCHS * (len(DATA[DATA_NAME]['train']) // BATCH_SIZE)    # dur 100 x 495 = 49.500; cok 100 x 985 = 98.500
EVERY = 5000                # kullanici: "her 5.000 adımda ölçme ve yedek olsun kesilirse kaldığı yerden devam eder"
SAVE_EVERY = 5000
ANSWER_ONLY = True          # kayip yalniz cevap rakamlari + son <eos> (eski tabanlar gibi); False: butun dizi
RESUME = False              # True: kosu kesildiyse (cekirdek dustu) once HAZIRLIK, sonra bu hucre -> son paketten devam

# --- GPU KAPISI (CLAUDE.md kural 2)
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, "GPU'da sadece %.1f GB bos" % _bos
print('GPU kapisi GECTI: %s  bos %.1f GB' % (torch.cuda.get_device_name(0), _bos))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme: 2 NABIZ
OUT = ROOT + '/' + NAME
C.start(NAME, DATA[DATA_NAME], OUT, steps=STEPS, seed=SEED, every=EVERY, device='cuda', setting=SETTING,
        save_every=SAVE_EVERY, resume=RESUME, batch_size=BATCH_SIZE, answer_only=ANSWER_ONLY, model_kw=MODEL_KW)
# compile varsayilan ACIK (kullanici, 27 Eylul: "bu sabit ayar ve yes olsun"); tarif varsayilani Muon + WSD (train_20)
print('basladi: %s  %d adim (%d epok)  batch %d  ->  %s' % (NAME, STEPS, EPOCHS, BATCH_SIZE, OUT))

In [ ]:
# 1b KOSU -- Model X, cok, D 96  |  GPU  |  tekrar: RESUME=False iken ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ; RESUME=True iken son checkpoint'ten SURDURUR
DATA_NAME, D = 'cok', 96
assert DATA_NAME == 'cok', "matematik yalniz cok (kullanici, 27 Eylul: 'matematik verisi sadece çok olsun')"
SETTING = 'shared'          # 'transformer_novalue': kiyas transformer'i (o zaman MODEL_KW'de turns yerine layers=2)
SEED = 0
NAME = 'modelx_math_%s_d%d_s%d' % (DATA_NAME, D, SEED)     # ONERI -- ad kullanici onayiyla (kural 11)
MODEL_KW = dict(d=D, units=4 * D, turns=2)
BATCH_SIZE = 256            # kullanici karari
EPOCHS = 100                # kullanici karari
STEPS = EPOCHS * (len(DATA[DATA_NAME]['train']) // BATCH_SIZE)    # dur 100 x 495 = 49.500; cok 100 x 985 = 98.500
EVERY = 5000                # kullanici: "her 5.000 adımda ölçme ve yedek olsun kesilirse kaldığı yerden devam eder"
SAVE_EVERY = 5000
ANSWER_ONLY = True          # kayip yalniz cevap rakamlari + son <eos> (eski tabanlar gibi); False: butun dizi
RESUME = False              # True: kosu kesildiyse (cekirdek dustu) once HAZIRLIK, sonra bu hucre -> son paketten devam

# --- GPU KAPISI (CLAUDE.md kural 2)
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, "GPU'da sadece %.1f GB bos" % _bos
print('GPU kapisi GECTI: %s  bos %.1f GB' % (torch.cuda.get_device_name(0), _bos))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme: 2 NABIZ
OUT = ROOT + '/' + NAME
C.start(NAME, DATA[DATA_NAME], OUT, steps=STEPS, seed=SEED, every=EVERY, device='cuda', setting=SETTING,
        save_every=SAVE_EVERY, resume=RESUME, batch_size=BATCH_SIZE, answer_only=ANSWER_ONLY, model_kw=MODEL_KW)
# compile varsayilan ACIK (kullanici, 27 Eylul: "bu sabit ayar ve yes olsun"); tarif varsayilani Muon + WSD (train_20)
print('basladi: %s  %d adim (%d epok)  batch %d  ->  %s' % (NAME, STEPS, EPOCHS, BATCH_SIZE, OUT))

In [ ]:
# 2 NABIZ  |  CPU  |  tekrar: GUVENLI
C.pulse(5)

In [ ]:
# 3 SONUC  |  CPU  |  tekrar: GUVENLI
# SAYI + METIN (kural 12): saglik, butun kumede terim ve hane kirilimi (kayit.txt ile ayni tablo), tani sorulari, yanlislar.
import json
RUN_NAMES = ('modelx_math_cok_d64_s0', 'modelx_math_cok_d96_s0')
for _name in RUN_NAMES:
    _p = ROOT + '/' + _name + '/final.json'
    if not os.path.exists(_p):
        print(_name, ': final.json yok')
        continue
    _f = json.load(open(_p, encoding='utf-8'))
    print('== %s  saglik (egitim ornegi %d) %.4f, esik %.2f' % (_name, _f['health']['n'], _f['health']['accuracy'], EM.HEALTH))
    for _side in ('train', 'heldout'):
        _v = _f[_side]
        print('  %-8s %.4f  uzunluk %.4f  ilk rakam %.4f  n %d' % (_side, _v['accuracy'], _v['length_ok'], _v['first_digit'], _v['n']))
        for _m in ('terms', 'digits'):
            print('     %-7s %s' % (_m, '   '.join('%s: %.4f (n %d)' % (k, v['accuracy'], v['n'])
                                                 for k, v in _f['breakdown'][_side][_m].items())))
    print('  tani sorulari (model_19 ile ayni 8 soru):')
    for _q, _want, _said, _ok in _f['panel']:
        print('     %-14s = %-5s model: %-12s %s' % (_q, _want, _said, 'DOGRU' if _ok else 'yanlis'))
    _wrong = [r for r in _f['rows'] if not r[3]]
    print('  tutulan ornegin ilk %d sorusunda yanlis %d; ilk 12:' % (len(_f['rows']), len(_wrong)))
    for _q, _want, _said, _ok in _wrong[:12]:
        print('     %-18s = %-5s model: %s' % (_q, _want, _said))

In [ ]:
# X DURDUR  |  CPU  |  tekrar: GUVENLI
# Bayrak koyar; iplik bir sonraki sinavda (en gec 5.000 adim sonra) modeli Drive'a kaydedip cikar.
C.stop()

In [ ]:
# Y DRIVE'DAKI KAYIT  |  CPU  |  tekrar: GUVENLI
for _n in sorted(os.listdir(ROOT)):
    _d = ROOT + '/' + _n
    if os.path.isdir(_d):
        _fs = sorted(os.listdir(_d))
        print('%-32s %s' % (_n, ' '.join(_fs)))

In [ ]:
# Z GPU DURUMU  |  CPU  |  tekrar: GUVENLI
!nvidia-smi